In [1]:
# CUDA Memory Management using Local and Shared Memory in Python (Numba)

import numpy as np
from numba import cuda, float32
import math

# CUDA Kernel
@cuda.jit
def memory_demo(input_array, output_array):

    # Shared Memory (shared by threads in a block)
    shared_mem = cuda.shared.array(shape=256, dtype=float32)

    # Thread index
    tid = cuda.threadIdx.x

    # Global index
    idx = cuda.grid(1)

    # ---------------- LOCAL MEMORY ----------------
    # Local variable (private to each thread)
    local_var = 0.0

    if idx < input_array.size:

        # Store data into local memory
        local_var = input_array[idx] * 2

        # Store into shared memory
        shared_mem[tid] = local_var

    # Synchronize threads in block
    cuda.syncthreads()

    # Read from shared memory
    if idx < input_array.size:

        output_array[idx] = shared_mem[tid] + 5


# Driver Code
if __name__ == "__main__":

    N = 256

    # Input Data
    input_data = np.arange(N).astype(np.float32)

    # Output Array
    output_data = np.zeros_like(input_data)

    # Copy data to GPU
    d_input = cuda.to_device(input_data)
    d_output = cuda.to_device(output_data)

    # CUDA Configuration
    threads_per_block = 256
    blocks_per_grid = math.ceil(N / threads_per_block)

    # Launch Kernel
    memory_demo[blocks_per_grid, threads_per_block](d_input, d_output)

    # Copy result back to CPU
    result = d_output.copy_to_host()

    print("Input Array:")
    print(input_data[:10])

    print("\nOutput Array:")
    print(result[:10])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Input Array:
[0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]

Output Array:
[ 5.  7.  9. 11. 13. 15. 17. 19. 21. 23.]
